# Tutorial 6: Precomputed PPI Network Embeddings

This notebook demonstrates how to load and use **precomputed protein-protein
interaction (PPI) embeddings** from the
[SPACE](https://doi.org/10.5281/zenodo.15600639) project, which covers
**1,322 eukaryotic species** in STRING 12.0.

Two types of embeddings are available:

| Type | Method | Dimensions | Description |
|---|---|---|---|
| `functional` | SPACE | 512 | Cross-species functional network embeddings |
| `node2vec` | node2vec | 128 | Single-species structural network embeddings |

Unlike sequence-based models (ESM2, Enformer, …), PPI embeddings capture
**functional relationships** between genes: two genes that share many
interaction partners will have similar embeddings, even if their sequences
are unrelated.

**Requirements**: only `h5py` is needed:
```bash
pip install h5py
```

In [4]:
import logging
logging.basicConfig(level=logging.INFO)

In [5]:
import torch
import numpy as np

from embpy.models.ppi_models import PrecomputedPPIWrapper

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## 1. Load functional (SPACE) embeddings

Point `data_dir` to the root of the precomputed embeddings directory.
The wrapper loads the HDF5 file and maps STRING protein accessions to
gene names via the STRING REST API automatically.

In [7]:
DATA_DIR = "/lustre/groups/ml01/workspace/goncalo.pinto/embpy/data/precomputed_embeddings_string"

wrapper = PrecomputedPPIWrapper(
    data_dir=DATA_DIR,
    species=2711,            # Citrus sinensis (example species with smaller file)
    embedding_type="functional",
)
wrapper.load(device)

print(f"Proteins loaded: {wrapper.num_proteins}")
print(f"Embedding dim:   {wrapper.embedding_dim}")
print(f"Gene names:      {len(wrapper.available_genes)}")
print(f"First 10 genes:  {wrapper.available_genes[:10]}")

INFO:embpy.models.ppi_models:Loaded functional embeddings for species 2711: 28114 proteins, dim=512


HTTPError: 522 Server Error: <none> for url: https://version-12-0.string-db.org/api/tsv/get_string_ids

## 2. Embed a single gene

Look up an embedding by gene symbol. You can also use the full STRING
protein ID (e.g. `"2711.A0A067E719"`).

In [ ]:
query_gene = wrapper.available_genes[0]
emb = wrapper.embed(query_gene)

print(f"{query_gene} embedding: shape={emb.shape}, dtype={emb.dtype}")
print(f"First 10 dims: {emb[:10]}")
print(f"L2 norm: {np.linalg.norm(emb):.2f}")

## 3. Embed a batch of genes

Genes not found in the mapping are returned as zero vectors (with a warning).

In [ ]:
query_genes = wrapper.available_genes[:5]
embs = wrapper.embed_batch(query_genes)

for gene, emb in zip(query_genes, embs):
    print(f"  {gene:12s} → dim {emb.shape[0]}, L2={np.linalg.norm(emb):.2f}")

## 4. Load node2vec embeddings

The same wrapper supports `embedding_type="node2vec"` for the 128-dim
single-species structural embeddings.

In [ ]:
n2v_wrapper = PrecomputedPPIWrapper(
    data_dir=DATA_DIR,
    species=2711,
    embedding_type="node2vec",
)
n2v_wrapper.load(device)

print(f"Node2vec proteins: {n2v_wrapper.num_proteins}")
print(f"Node2vec dim:      {n2v_wrapper.embedding_dim}")

gene = n2v_wrapper.available_genes[0]
n2v_emb = n2v_wrapper.embed(gene)
print(f"\n{gene} node2vec embedding: shape={n2v_emb.shape}")

## 5. Explore PPI similarity

Genes with similar interaction profiles should have high cosine similarity.

In [ ]:
import matplotlib.pyplot as plt
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-10)

sample_genes = wrapper.available_genes[:30]
sample_embs = np.array(wrapper.embed_batch(sample_genes))

sim_matrix = np.zeros((len(sample_genes), len(sample_genes)))
for i in range(len(sample_genes)):
    for j in range(len(sample_genes)):
        sim_matrix[i, j] = cosine_sim(sample_embs[i], sample_embs[j])

print(f"Similarity matrix shape: {sim_matrix.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(sample_genes)))
ax.set_yticks(range(len(sample_genes)))
ax.set_xticklabels(sample_genes, rotation=90, fontsize=7)
ax.set_yticklabels(sample_genes, fontsize=7)
ax.set_title("Cosine similarity – SPACE functional embeddings")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 6. 2D PCA visualisation

In [ ]:
from sklearn.decomposition import PCA

pca_2d = PCA(n_components=2).fit_transform(sample_embs)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(pca_2d[:, 0], pca_2d[:, 1], s=80, alpha=0.8, edgecolors="k", linewidths=0.5)
for i, gene in enumerate(sample_genes):
    ax.annotate(gene, (pca_2d[i, 0], pca_2d[i, 1]), fontsize=7, ha="left", va="bottom")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA of SPACE functional embeddings")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Combining PPI embeddings with sequence embeddings

PPI embeddings capture **functional context** while sequence embeddings
(ESM2, Enformer, etc.) capture **molecular structure**. Combining both
in an AnnData object provides a richer representation.

In [ ]:
import anndata as ad
import pandas as pd

genes = wrapper.available_genes[:50]
ppi_matrix = np.array(wrapper.embed_batch(genes))

adata_ppi = ad.AnnData(
    X=np.zeros((len(genes), 1)),
    obs=pd.DataFrame({"gene_name": genes}, index=genes),
)
adata_ppi.obsm["X_ppi_functional"] = ppi_matrix

print(adata_ppi)
print(f"\nPPI embeddings stored in adata.obsm['X_ppi_functional']: {adata_ppi.obsm['X_ppi_functional'].shape}")

In [ ]:
from embpy.pp.basic import reduce_embeddings

adata_ppi = reduce_embeddings(adata_ppi, obsm_key="X_ppi_functional", n_components=50)
print(f"PCA-reduced embeddings: {adata_ppi.obsm['X_ppi_functional_pca'].shape}")

## Summary

| Step | Code |
|---|---|
| Create wrapper | `PrecomputedPPIWrapper(data_dir=..., species=9606, embedding_type="functional")` |
| Load embeddings | `wrapper.load(device)` |
| Embed one gene | `wrapper.embed("TP53")` |
| Embed batch | `wrapper.embed_batch(["TP53", "BRCA1", ...])` |
| Node2vec variant | `PrecomputedPPIWrapper(..., embedding_type="node2vec")` |
| Available genes | `wrapper.available_genes` |
| Available proteins | `wrapper.available_proteins` |
| Embedding dimension | `wrapper.embedding_dim` |